# Day 5: Evaluation

https://docs.google.com/document/d/1laKd7zWd9P6vgM06gIQMX7_EHdaPy43o5qIfz_6-zgE/edit?tab=t.0#heading=h.6f40algdx96s

In [ ]:
from utils.ingest import read_repo_data
from minsearch import Index

# DataTalksClub FAQ
dtc_faq = read_repo_data('DataTalksClub', 'faq')

de_dtc_faq = [d for d in dtc_faq if 'data-engineering' in d['filename']]

faq_index = Index(
    text_fields=["question", "content"],
    keyword_fields=[]
)

faq_index.fit(de_dtc_faq)

In [ ]:
from typing import List, Any
from pydantic_ai import Agent


def text_search(query: str) -> List[Any]:
    """
    Perform a text-based search on the FAQ index.

    Args:
        query (str): The search query string.

    Returns:
        List[Any]: A list of up to 5 search results returned by the FAQ index.
    """
    return faq_index.search(query, num_results=5)


system_prompt = """
You are a helpful assistant for a  course. 

Use the search tool to find relevant information from the course materials before answering questions.

If you can find specific information through search, use it to provide accurate answers.
If the search doesn't return relevant results, let the user know and provide general guidance.
"""

from pydantic_ai import Agent

agent = Agent(
    name="faq_agent",
    instructions=system_prompt,
    tools=[text_search],
    model='groq:openai/gpt-oss-20b'
)


In [ ]:
question = "how do I install Kafka in Python?"
result = await agent.run(user_prompt=question)

In [ ]:
print(result.output)

### Logging

We want to capture the following information from each agent run:

- **System prompt** used  
- **Model** chosen  
- **User query**  
- **Tools** invoked  
- **LLM ? tool interactions** (inputs/outputs)  
- **Final response**  

---

#### Simple Approach
- Implement a minimal logging system.  
- Save logs as **JSON files** for easy review.  

---

#### ?? Important
- This is **not production-ready**.  
- In practice, send logs to a log collection system or use specialized LLM evaluation tools such as:  
  - **Evidently**  
  - **LangWatch**  
  - **Arize Phoenix**  

---

#### Example JSON Log Structure
```json
{
  "agent_name": "support_agent",
  "system_prompt": "You are a helpful assistant.",
  "provider": "groq",
  "model": "openai/gpt-oss-20b",
  "tools": ["text_search"],
  "messages": [
    {
      "role": "user",
      "content": "What is scaffolding in education?"
    },
    {
      "role": "assistant",
      "content": "Scaffolding in education means providing structured support..."
    }
  ],
  "source": "user"
}
```




In [ ]:
from pydantic_ai.messages import ModelMessagesTypeAdapter


def log_entry(agent, messages, source="user"):
    tools = []

    for ts in agent.toolsets:
        tools.extend(ts.tools.keys())

    # Convert internal message format into regular Python dictionaries
    dict_messages = ModelMessagesTypeAdapter.dump_python(messages)

    return {
        "agent_name": agent.name,
        "system_prompt": agent._instructions,
        "provider": agent.model.system,
        "model": agent.model.model_name,
        "tools": tools,
        "messages": dict_messages,
        "source": source
    }


In [ ]:
import json
import secrets
from pathlib import Path
from datetime import datetime

# Create logs directory if it doesn't exist
LOG_DIR = Path('logs')
LOG_DIR.mkdir(exist_ok=True)


# Custom serializer for datetime objects
def serializer(obj):
    if isinstance(obj, datetime):
        return obj.isoformat()
    raise TypeError(f"Type {type(obj)} not serializable")


def log_interaction_to_file(agent, messages, source='user'):
    entry = log_entry(agent, messages, source)

    # Generate a unique filename with timestamp and random hex
    ts = entry['messages'][-1]['timestamp']
    ts_str = ts.strftime("%Y%m%d_%H%M%S")
    rand_hex = secrets.token_hex(3)

    filename = f"{agent.name}_{ts_str}_{rand_hex}.json"
    filepath = LOG_DIR / filename

    # Save the completed logs to a JSON file
    with filepath.open("w", encoding="utf-8") as f_out:
        json.dump(entry, f_out, indent=2, default=serializer)

    return filepath

In [ ]:
# Create a simple interaction loop
question = input()
result = await agent.run(user_prompt=question)
print(result.output)
log_interaction_to_file(agent, result.new_messages())

In [ ]:
result.new_messages()

### Adding References

Add references to the original documents in the agent?s responses.

In [ ]:
system_prompt = """
You are a helpful assistant for a course.  

Use the search tool to find relevant information from the course materials before answering questions.  

If you can find specific information through search, use it to provide accurate answers.

Always include references by citing the filename of the source material you used.  
When citing the reference, replace "faq-main" by the full path to the GitHub repository: "https://github.com/DataTalksClub/faq/blob/main/"
Format: [LINK TITLE](FULL_GITHUB_LINK)

If the search doesn't return relevant results, let the user know and provide general guidance.  
""".strip()

# Create another version of agent, let's call it faq_agent_v2
agent = Agent(
    name="faq_agent_v2",
    instructions=system_prompt,
    tools=[text_search],
    model='groq:openai/gpt-oss-20b'
)


In [ ]:
question = "can I join late and get a certificate?"
result = await agent.run(user_prompt=question)
print(result.output)
log_interaction_to_file(agent, result.new_messages())

In [ ]:
result.new_messages()

### Evaluation

- **Vibe Check**:  
  - A quick manual review to get a sense of how well the model is performing.  
  - Collect ~10?20 examples for inspection.  
  - Helps identify edge cases and define future evaluation criteria.  

- **LLM as a Judge**:  
  - Use one LLM to evaluate another?s outputs.  
  - Checks can include:  
    - Following instructions  
    - Answer quality/relevance  
    - References included  
    - Proper tool usage  
  - Automates evaluation once criteria are defined.  


In [ ]:
evaluation_prompt = """
Use this checklist to evaluate the quality of an AI agent's answer (<ANSWER>) to a user question (<QUESTION>).
We also include the entire log (<LOG>) for analysis.

For each item, check if the condition is met. 

Checklist:

- instructions_follow: The agent followed the user's instructions (in <INSTRUCTIONS>)
- instructions_avoid: The agent avoided doing things it was told not to do  
- answer_relevant: The response directly addresses the user's question  
- answer_clear: The answer is clear and correct  
- answer_citations: The response includes proper citations or sources when required  
- completeness: The response is complete and covers all key aspects of the request
- tool_call_search: The agent invoked search tool (in <LOG>)

Only fill true/false and justification. Do not include any explanation or extra fields.
""".strip()

#### Structured Output

- Use structured output when expecting a well-defined response format.  
- Define a **Pydantic class** with the expected schema.  
- The LLM will generate output that matches the schema exactly, ensuring consistency and easier parsing.  


In [ ]:
from pydantic import BaseModel, Field

In [ ]:
class EvaluationCheck(BaseModel):
    check_name: str = Field(description="The name of the check from the provided checklist.")
    justification: str = Field(description="A brief explanation for the check_pass result.")
    check_pass: bool = Field(description="True if the condition is met, False otherwise.")

class EvaluationChecklist(BaseModel):
    checklist: list[EvaluationCheck]
    summary: str = Field(description="A concise final summary of the agent's performance.")


In [ ]:
# Create an evaluation agent with structured output
eval_agent = Agent(
    name='eval_agent',
    model='groq:llama-3.3-70b-versatile', # Use the model that can handle structured output
    instructions=evaluation_prompt,
    output_type=EvaluationChecklist
)


In [ ]:
# Input format for the evaluation agent - use XML-like tags for better understanding
user_prompt_format = """
<INSTRUCTIONS>{instructions}</INSTRUCTIONS>
<QUESTION>{question}</QUESTION>
<ANSWER>{answer}</ANSWER>
<LOG>{log}</LOG>
""".strip()

In [ ]:
# Helper function to load log file
def load_log_file(log_file):
    with open(log_file, 'r') as f_in:
        log_data = json.load(f_in)
        log_data['log_file'] = log_file
        return log_data

In [ ]:
# Loads a saved interaction log
log_record = load_log_file('./logs/faq_agent_v2_20251002_115904_1cd9e6.json')

# Extracts the key components (instructions, question, answer, full log)
instructions = log_record['system_prompt'][0]
question = log_record['messages'][0]['parts'][0]['content']
answer = log_record['messages'][-1]['parts'][-1]['content']
log = json.dumps(log_record['messages'])

print("Instructions:", instructions)
print("Question:", question)
print("Answer:", answer)
print("Log:", log[:500], "...")  # Print only the first 500 characters

In [ ]:
# Formats them into the evaluation prompt
user_prompt = user_prompt_format.format(
    instructions=instructions,
    question=question,
    answer=answer,
    log=log
)

print("User Prompt:", user_prompt[:500], "...")  # Print only the first 500 characters

In [ ]:
# Runs the evaluation agent
result = await eval_agent.run(user_prompt, output_type=EvaluationChecklist)

checklist = result.output
print(checklist.summary)

for check in checklist.checklist:
    print(check)

#### Reduce prompt verbosity: 

Keep the prompt concise by including only the necessary context, rather than the entire conversation log.

- remove timestamps and IDs that aren't needed for evaluation
- remove thinking part that are not necessary
- replace actual search results with a placeholder
- keep only the essential structure

This reduces the number of tokens we send to the evaluation model, which lowers the costs and speeds up evaluation.

In [ ]:
def simplify_log_messages(messages):
    log_simplified = []

    for m in messages:
        parts = []
    
        for original_part in m['parts']:
            part = original_part.copy()
            kind = part['part_kind']
            
            if kind == 'thinking':
                part['content'] = 'THINKING_REDACTED'
            if kind == 'user-prompt':
                del part['timestamp']
            if kind == 'tool-call':
                del part['tool_call_id']
            if kind == 'tool-return':
                del part['tool_call_id']
                del part['metadata']
                del part['timestamp']
                # Replace actual search results with placeholder to save tokens
                part['content'] = 'RETURN_RESULTS_REDACTED'
            if kind == 'text':
                del part['id']
    
            parts.append(part)
    
        message = {
            'kind': m['kind'],
            'parts': parts
        }
    
        log_simplified.append(message)
    return log_simplified

In [ ]:
simple_log = simplify_log_messages(log_record['messages'])
print(json.dumps(simple_log, indent=2))  # Print only

In [ ]:
async def evaluate_log_record(eval_agent, log_record):
    messages = log_record['messages']

    instructions = log_record['system_prompt']
    question = messages[0]['parts'][0]['content']
    answer = messages[-1]['parts'][-1]['content']

    log_simplified = simplify_log_messages(messages)
    log = json.dumps(log_simplified)

    user_prompt = user_prompt_format.format(
        instructions=instructions,
        question=question,
        answer=answer,
        log=log
    )

    result = await eval_agent.run(user_prompt, output_type=EvaluationChecklist)
    return result.output 


log_record = load_log_file('./logs/faq_agent_v2_20251002_115904_1cd9e6.json')
eval1 = await evaluate_log_record(eval_agent, log_record)

In [ ]:
print(eval1)

### Data Generation

- Use AI to generate additional questions from sample records in the database.  
- For each record:  
  1. Ask an LLM to create a question based on the record.  
  2. Use the generated question as input to the agent and log the response.  
- Current approach is simple; more advanced methods could also track the source file for later verification.  
- Adjust prompts according to your specific project or use case.  

In [ ]:
question_generation_prompt = """
You are helping to create test questions for an AI agent that answers questions about a data engineering course.

Based on the provided FAQ content, generate realistic questions that students might ask.

The questions should:

- Be natural and varied in style
- Range from simple to complex
- Include both specific technical questions and general course questions

Generate one question for each record.
""".strip()

class QuestionsList(BaseModel):
    questions: list[str]

question_generator = Agent(
    name="question_generator",
    instructions=question_generation_prompt,
    model='groq:openai/gpt-oss-20b',
    output_type=QuestionsList
)


In [ ]:
import random

# Randomly sample 10 records from the FAQ data
sample = random.sample(de_dtc_faq, 10)
prompt_docs = [d['content'] for d in sample]

In [ ]:
prompt_docs

In [ ]:
# Generate questions
prompt = json.dumps(prompt_docs)

result = await question_generator.run(prompt)
questions = result.output.questions

In [ ]:
questions

In [ ]:
from tqdm.auto import tqdm

# Use the generated questions as input to the agent and log the response
for q in tqdm(questions):
    print(q)

    result = await agent.run(user_prompt=q)
    print(result.output)

    log_interaction_to_file(
        agent,
        result.new_messages(),
        source='ai-generated'
    )

    print()

- Repeat generation until sufficient data is collected (e.g., ~100 examples).  
- For now, we can use the 10 existing log records.  
- Benefits of AI-generated data:  
  - Faster data creation  
  - Can cover edge cases that might be overlooked  
- Limitations:  
  - May not reflect real user behavior  
  - Might miss edge cases only real users encounter  
  - May not capture full complexity of real queries 

In [ ]:
# Collect all AI-generated logs for evaluation
eval_set = []

for log_file in LOG_DIR.glob('*.json'):
    if 'faq_agent_v2' not in log_file.name:
        continue

    log_record = load_log_file(log_file)
    if log_record['source'] != 'ai-generated':
        continue

    eval_set.append(log_record)

In [ ]:
len(eval_set)

In [ ]:
eval_results = []

for log_record in tqdm(eval_set):
    eval_result = await evaluate_log_record(eval_agent, log_record)
    print(eval_result)
    eval_results.append((log_record, eval_result)) # Store both log and evaluation result

In [ ]:
# Transform results into a tabular format for analysis
rows = []

for log_record, eval_result in eval_results:
    messages = log_record['messages']

    row = {
        'file': log_record['log_file'].name,
        'question': messages[0]['parts'][0]['content'],
        'answer': messages[-1]['parts'][0]['content'],
    }

    checks = {c.check_name: c.check_pass for c in eval_result.checklist}
    row.update(checks)

    rows.append(row)

In [ ]:
import pandas as pd

df_evals = pd.DataFrame(rows)
df_evals.head()

In [ ]:
df_evals.mean(numeric_only=True)

### Evaluating Functions and Tools

- Tools should be evaluated **separately from the agent**.  
- For code tools: use **unit and integration tests**.  
- For search functions: evaluate with **information retrieval metrics**, such as:  
  - **Precision & Recall**: fraction of relevant results retrieved vs. missed  
  - **Hit Rate**: % of queries returning at least one relevant result  
  - **MRR (Mean Reciprocal Rank)**: position of the first relevant result in ranking  
- Implement Hit Rate and MRR calculations in Python for automated evaluation. 

In [ ]:
def evaluate_search_quality(search_function, test_queries):
    results = []
    
    for query, expected_docs in test_queries:
        search_results = search_function(query)
        
        # Calculate hit rate
        relevant_found = any(doc['filename'] in expected_docs for doc in search_results)
        
        # Calculate MRR
        for i, doc in enumerate(search_results):
            if doc['filename'] in expected_docs:
                mrr = 1 / (i + 1)
                break
        else:
            mrr = 0
            
        results.append({
            'query': query,
            'hit': relevant_found,
            'mrr': mrr
        })
    return results

In [ ]:
sample[:2]

In [ ]:
tests_queries = [(item['question'], [item['filename']]) for item in sample]
tests_queries[:2]

In [ ]:
test_results = evaluate_search_quality(text_search, tests_queries)

In [ ]:
hit_rate = sum(r['hit'] for r in test_results) / len(test_results) if test_results else 0
avg_mrr = sum(r['mrr'] for r in test_results) / len(test_results) if test_results else 0

In [ ]:
print(f"Hit Rate: {hit_rate:.2f}")
print(f"Average MRR: {avg_mrr:.2f}")